# Circuit Breaker Pattern - Complete Guide for Beginners

## What is a Circuit Breaker?

**Real-world analogy**: Think of a circuit breaker in your home electricity panel.
- When there's too much electrical current (danger!), the circuit breaker "trips" (opens)
- It stops electricity flow to protect your appliances
- After some time, you can reset it to try again

**In Software**: A Circuit Breaker is a design pattern that:
- **Protects your application** from repeatedly calling a failing service
- **Prevents cascading failures** - when one service fails, it doesn't bring down your whole system
- **Saves resources** - stops wasting time/money on requests that will fail
- **Auto-recovers** - automatically tries again after some time


## Real-World Example

**Scenario**: Your e-commerce website depends on:
- Payment Service API
- Inventory Service API
- Shipping Service API

**Problem**: If Payment Service is down (maybe their database crashed):
- Every user checkout tries to call Payment Service
- Each call takes 30 seconds to timeout
- 1000 users → 1000 × 30 seconds = 30,000 seconds wasted!
- Your server gets overwhelmed and crashes too 😱

**Solution**: Circuit Breaker!
- After 5 failures, "open" the circuit (stop calling Payment Service)
- Immediately return error to user (no 30-second wait)
- After 60 seconds, try again (maybe service is fixed)
- If it works, "close" the circuit (normal operation resumes)


## Circuit Breaker States

A Circuit Breaker has **3 states**:

### 1. **CLOSED** (Normal State) ✅
- **Meaning**: Circuit is "closed" = current flows = requests go through normally
- **Behavior**: All requests are sent to the service
- **Tracking**: Counts failures (if service fails, increment failure count)
- **Transition**: If failures exceed threshold (e.g., 5 failures), move to OPEN state

### 2. **OPEN** (Failing State) 🔴
- **Meaning**: Circuit is "open" = current blocked = requests are rejected immediately
- **Behavior**: Requests are rejected immediately without calling the service
- **Timer**: Waits for a timeout period (e.g., 60 seconds)
- **Transition**: After timeout, move to HALF-OPEN state (test if service is back)

### 3. **HALF-OPEN** (Testing State) 🟡
- **Meaning**: Testing if service is recovered
- **Behavior**: Allows limited requests (e.g., 1-3 requests) to test the service
- **Success**: If requests succeed, move back to CLOSED (service is healthy)
- **Failure**: If requests fail, move back to OPEN (service still broken)

```
         CLOSED (Normal)
            ↓
    (5 failures detected)
            ↓
        OPEN (Blocking)
            ↓
    (60 seconds timeout)
            ↓
     HALF-OPEN (Testing)
       ↙        ↘
  SUCCESS     FAILURE
     ↓           ↓
  CLOSED       OPEN
```


## Step-by-Step: How It Works

### Example Timeline:

1. **Time 0:00** - Circuit is CLOSED
   - Request 1 → Service works ✅
   - Request 2 → Service works ✅

2. **Time 0:05** - Service starts failing
   - Request 3 → Service fails ❌ (failure count: 1)
   - Request 4 → Service fails ❌ (failure count: 2)
   - Request 5 → Service fails ❌ (failure count: 3)
   - Request 6 → Service fails ❌ (failure count: 4)
   - Request 7 → Service fails ❌ (failure count: 5) → **THRESHOLD REACHED!**

3. **Time 0:06** - Circuit OPENS 🔴
   - Request 8 → **Rejected immediately** (no service call, returns error instantly)
   - Request 9 → **Rejected immediately**
   - Request 10 → **Rejected immediately**
   - All future requests rejected until timeout...

4. **Time 1:06** - 60 seconds passed, Circuit goes HALF-OPEN 🟡
   - Request 11 → **Test request** sent to service
   - Service still down → **FAILURE** → Circuit goes back to OPEN

5. **Time 2:06** - Another 60 seconds passed, Circuit goes HALF-OPEN again
   - Request 12 → **Test request** sent to service
   - Service is back! → **SUCCESS** ✅ → Circuit goes to CLOSED

6. **Time 2:07** - Circuit is CLOSED again ✅
   - Request 13 → Service works ✅
   - Normal operation resumes


## Python Implementation

Let's build a simple Circuit Breaker from scratch!


In [8]:
# Step 1: Define the states using an Enum (enumeration)
# An Enum is like a set of named constants
from enum import Enum
import time

class CircuitState(Enum):
    """Three states of circuit breaker"""
    CLOSED = "CLOSED"      # Normal operation - requests go through
    OPEN = "OPEN"          # Service is failing - reject requests immediately
    HALF_OPEN = "HALF_OPEN" # Testing if service recovered
    
print("States available:")
for state in CircuitState:
    print(f"  - {state.name}: {state.value}")


States available:
  - CLOSED: CLOSED
  - OPEN: OPEN
  - HALF_OPEN: HALF_OPEN


In [9]:
# Step 2: Create the Circuit Breaker class
class CircuitBreaker:
    """
    Simple Circuit Breaker implementation
    
    Parameters:
    - failure_threshold: How many failures before opening circuit (default: 5)
    - timeout: How long to wait before trying again (default: 60 seconds)
    - half_open_max_calls: How many test calls in HALF_OPEN state (default: 3)
    """
    
    def __init__(self, failure_threshold=5, timeout=60, half_open_max_calls=3):
        # Configuration
        self.failure_threshold = failure_threshold  # Open circuit after 5 failures
        self.timeout = timeout  # Wait 60 seconds before testing again
        self.half_open_max_calls = half_open_max_calls  # Allow 3 test calls
        
        # State tracking
        self.state = CircuitState.CLOSED  # Start in CLOSED state
        self.failure_count = 0  # Count consecutive failures
        self.last_failure_time = None  # When did last failure happen?
        self.half_open_calls = 0  # How many calls made in HALF_OPEN state
        
    def call(self, func, *args, **kwargs):
        """
        Call a function through the circuit breaker
        
        Args:
            func: The function to call (e.g., API call)
            *args, **kwargs: Arguments to pass to the function
        
        Returns:
            Result from the function if successful
        
        Raises:
            CircuitBreakerOpenError: If circuit is OPEN
            Original exception: If function call fails
        """
        
        # Check if we should transition states
        self._check_and_transition_state()
        
        # Handle based on current state
        if self.state == CircuitState.OPEN:
            raise CircuitBreakerOpenError(
                f"Circuit breaker is OPEN. Service unavailable. "
                f"Will retry after {self.timeout} seconds."
            )
        
        elif self.state == CircuitState.HALF_OPEN:
            # In HALF_OPEN, we allow limited calls for testing
            if self.half_open_calls >= self.half_open_max_calls:
                raise CircuitBreakerOpenError(
                    "Circuit breaker is HALF_OPEN. Max test calls reached."
                )
            self.half_open_calls += 1
        
        # Try to call the function
        try:
            result = func(*args, **kwargs)
            # Success! Reset failure count and change state
            self._on_success()
            return result
            
        except Exception as e:
            # Failure! Increment count and update state
            self._on_failure()
            raise e  # Re-raise the original exception
    
    def _check_and_transition_state(self):
        """Check if we need to change state (OPEN → HALF_OPEN based on timeout)"""
        if self.state == CircuitState.OPEN:
            # Check if timeout period has passed
            if self.last_failure_time and (time.time() - self.last_failure_time) >= self.timeout:
                print(f"⏰ Timeout ({self.timeout}s) passed. Moving to HALF_OPEN state for testing...")
                self.state = CircuitState.HALF_OPEN
                self.half_open_calls = 0  # Reset test call counter
    
    def _on_success(self):
        """Handle successful call"""
        self.failure_count = 0  # Reset failure count
        
        if self.state == CircuitState.HALF_OPEN:
            print("✅ Service is back! Moving to CLOSED state.")
            self.state = CircuitState.CLOSED
        
    def _on_failure(self):
        """Handle failed call"""
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.state == CircuitState.CLOSED:
            print(f"❌ Failure {self.failure_count}/{self.failure_threshold}")
            if self.failure_count >= self.failure_threshold:
                print(f"🔴 Circuit breaker OPENED after {self.failure_count} failures!")
                self.state = CircuitState.OPEN
                
        elif self.state == CircuitState.HALF_OPEN:
            print(f"❌ Test call failed. Moving back to OPEN state.")
            self.state = CircuitState.OPEN
            self.half_open_calls = 0
    
    def get_state(self):
        """Get current state of circuit breaker"""
        return self.state.value
    
    def reset(self):
        """Manually reset circuit breaker to CLOSED state"""
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = None
        self.half_open_calls = 0
        print("🔄 Circuit breaker manually reset to CLOSED state.")


# Custom exception for when circuit is open
class CircuitBreakerOpenError(Exception):
    """Exception raised when circuit breaker is OPEN"""
    pass

print("✅ CircuitBreaker class created!")


✅ CircuitBreaker class created!


## Example 1: Simulating a Failing Service

Let's create a mock service that fails randomly to test our circuit breaker!


In [10]:
import random

# Simulate a service that sometimes fails
class UnreliableService:
    def __init__(self, failure_rate=0.7):
        """
        failure_rate: Probability of failure (0.0 = never fails, 1.0 = always fails)
        """
        self.failure_rate = failure_rate
        self.call_count = 0
    
    def call_service(self):
        """Simulates calling an external API"""
        self.call_count += 1
        
        # Simulate network delay
        time.sleep(0.1)  # 100ms delay
        
        # Randomly succeed or fail
        if random.random() < self.failure_rate:
            raise Exception(f"Service error: Database connection failed (call #{self.call_count})")
        
        return f"Service response: Success! (call #{self.call_count})"

# Create service that fails 70% of the time
service = UnreliableService(failure_rate=0.7)

# Create circuit breaker
cb = CircuitBreaker(failure_threshold=3, timeout=5)  # Open after 3 failures, wait 5 seconds

print("Service and Circuit Breaker ready!")
print(f"Initial state: {cb.get_state()}")


Service and Circuit Breaker ready!
Initial state: CLOSED


In [11]:
# Test the circuit breaker
print("\n=== Testing Circuit Breaker ===\n")

for i in range(10):
    print(f"\nRequest {i+1}:")
    print(f"  Current state: {cb.get_state()}")
    
    try:
        # Call service through circuit breaker
        result = cb.call(service.call_service)
        print(f"  ✅ {result}")
        
    except CircuitBreakerOpenError as e:
        print(f"  🔴 {e}")
        
    except Exception as e:
        print(f"  ❌ {e}")
    
    time.sleep(0.5)  # Small delay between requests



=== Testing Circuit Breaker ===


Request 1:
  Current state: CLOSED
❌ Failure 1/3
  ❌ Service error: Database connection failed (call #1)

Request 2:
  Current state: CLOSED
  ✅ Service response: Success! (call #2)

Request 3:
  Current state: CLOSED
  ✅ Service response: Success! (call #3)

Request 4:
  Current state: CLOSED
❌ Failure 1/3
  ❌ Service error: Database connection failed (call #4)

Request 5:
  Current state: CLOSED
  ✅ Service response: Success! (call #5)

Request 6:
  Current state: CLOSED
  ✅ Service response: Success! (call #6)

Request 7:
  Current state: CLOSED
❌ Failure 1/3
  ❌ Service error: Database connection failed (call #7)

Request 8:
  Current state: CLOSED
❌ Failure 2/3
  ❌ Service error: Database connection failed (call #8)

Request 9:
  Current state: CLOSED
❌ Failure 3/3
🔴 Circuit breaker OPENED after 3 failures!
  ❌ Service error: Database connection failed (call #9)

Request 10:
  Current state: OPEN
  🔴 Circuit breaker is OPEN. Service unavailable. 

## Example 2: Real API Call Simulation

Let's simulate what happens with a real payment API that goes down:


In [12]:
# Simulate payment service
class PaymentService:
    def __init__(self, is_healthy=True):
        self.is_healthy = is_healthy
    
    def process_payment(self, amount):
        """Simulate processing a payment"""
        if not self.is_healthy:
            raise Exception("Payment service is down! Database connection failed.")
        
        # Simulate processing time
        time.sleep(0.2)
        return f"Payment of ${amount} processed successfully!"
    
    def set_health(self, healthy):
        """Manually set service health (for testing)"""
        self.is_healthy = healthy
        status = "healthy" if healthy else "down"
        print(f"🏥 Payment service status changed to: {status}")

# Create payment service (starts healthy, then we'll make it fail)
payment_service = PaymentService(is_healthy=True)

# Create circuit breaker for payment service
payment_cb = CircuitBreaker(failure_threshold=3, timeout=3)

print("=== Payment Service with Circuit Breaker ===\n")


=== Payment Service with Circuit Breaker ===



In [13]:
# Scenario: Service is healthy, then goes down, then recovers

print("Phase 1: Service is healthy ✅")
print("-" * 50)
for i in range(3):
    try:
        result = payment_cb.call(payment_service.process_payment, 100)
        print(f"Request {i+1}: {result}")
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")

print("\nPhase 2: Service goes down! 😱")
print("-" * 50)
payment_service.set_health(False)

for i in range(5):
    try:
        result = payment_cb.call(payment_service.process_payment, 100)
        print(f"Request {i+1}: {result}")
    except CircuitBreakerOpenError as e:
        print(f"Request {i+1}: 🔴 Circuit OPEN - {e}")
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")

print("\n⏳ Waiting for timeout period...")
time.sleep(4)  # Wait for circuit to go HALF_OPEN

print("\nPhase 3: Testing if service recovered...")
print("-" * 50)
try:
    result = payment_cb.call(payment_service.process_payment, 100)
    print(f"Test request: {result}")
except Exception as e:
    print(f"Test request: ❌ {e}")

print("\nPhase 4: Service is back! 🎉")
print("-" * 50)
payment_service.set_health(True)

time.sleep(4)  # Wait for next timeout

try:
    result = payment_cb.call(payment_service.process_payment, 100)
    print(f"Request after recovery: {result}")
except Exception as e:
    print(f"Request after recovery: ❌ {e}")


Phase 1: Service is healthy ✅
--------------------------------------------------
Request 1: Payment of $100 processed successfully!
Request 2: Payment of $100 processed successfully!
Request 3: Payment of $100 processed successfully!

Phase 2: Service goes down! 😱
--------------------------------------------------
🏥 Payment service status changed to: down
❌ Failure 1/3
Request 1: ❌ Payment service is down! Database connection failed.
❌ Failure 2/3
Request 2: ❌ Payment service is down! Database connection failed.
❌ Failure 3/3
🔴 Circuit breaker OPENED after 3 failures!
Request 3: ❌ Payment service is down! Database connection failed.
Request 4: 🔴 Circuit OPEN - Circuit breaker is OPEN. Service unavailable. Will retry after 3 seconds.
Request 5: 🔴 Circuit OPEN - Circuit breaker is OPEN. Service unavailable. Will retry after 3 seconds.

⏳ Waiting for timeout period...

Phase 3: Testing if service recovered...
--------------------------------------------------
⏰ Timeout (3s) passed. Moving

## Example 3: Comparing With and Without Circuit Breaker

Let's see the difference in performance:


In [14]:
import time

class SlowFailingService:
    def __init__(self):
        self.is_down = False
    
    def call(self):
        if self.is_down:
            time.sleep(3)  # Simulate 3-second timeout
            raise Exception("Service timeout")
        return "Success"

# Service that goes down after first call
service = SlowFailingService()
service.call()  # First call succeeds
service.is_down = True  # Then it goes down

print("=== WITHOUT Circuit Breaker ===")
print("Making 10 requests to failing service...")
start_time = time.time()

for i in range(10):
    try:
        service.call()
    except:
        pass  # Ignore errors

time_without_cb = time.time() - start_time
print(f"⏱️  Time taken: {time_without_cb:.2f} seconds (each request waited 3 seconds!)")

print("\n=== WITH Circuit Breaker ===")
service2 = SlowFailingService()
service2.call()  # First call succeeds
service2.is_down = True

cb2 = CircuitBreaker(failure_threshold=1, timeout=60)  # Open after 1 failure

print("Making 10 requests to failing service...")
start_time = time.time()

for i in range(10):
    try:
        cb2.call(service2.call)
    except CircuitBreakerOpenError:
        pass  # Circuit open - immediate rejection
    except:
        pass  # First failure

time_with_cb = time.time() - start_time
print(f"⏱️  Time taken: {time_with_cb:.2f} seconds (circuit opened, rejections are instant!)")

print(f"\n💰 Time saved: {time_without_cb - time_with_cb:.2f} seconds")
print(f"🚀 Speed improvement: {time_without_cb / time_with_cb:.1f}x faster")


=== WITHOUT Circuit Breaker ===
Making 10 requests to failing service...
⏱️  Time taken: 30.04 seconds (each request waited 3 seconds!)

=== WITH Circuit Breaker ===
Making 10 requests to failing service...
❌ Failure 1/1
🔴 Circuit breaker OPENED after 1 failures!
⏱️  Time taken: 3.00 seconds (circuit opened, rejections are instant!)

💰 Time saved: 27.04 seconds
🚀 Speed improvement: 10.0x faster


## Using pybreaker Library (Production-Ready)

Now that you understand how circuit breaker works, let's learn how to use `pybreaker` - a production-ready library!

**Why use a library instead of building from scratch?**
- ✅ Battle-tested in production
- ✅ More features (callbacks, listeners, state storage)
- ✅ Better error handling
- ✅ Thread-safe
- ✅ Less code to maintain

**Installation**: `pip install pybreaker`


In [15]:
# First, let's install pybreaker (if not already installed)
# Uncomment the line below if you need to install it
# !pip install pybreaker

import pybreaker
import time
from pybreaker import CircuitBreaker, CircuitBreakerError

print("✅ pybreaker imported successfully!")
print(f"pybreaker version: {pybreaker.__version__ if hasattr(pybreaker, '__version__') else 'installed'}")


ModuleNotFoundError: No module named 'pybreaker'

### Example 1: Basic Usage with pybreaker

Let's create a simple circuit breaker using pybreaker:


In [ ]:
# Create a circuit breaker with custom settings
cb = CircuitBreaker(
    fail_max=5,              # Open circuit after 5 failures (like failure_threshold)
    timeout_duration=60,     # Wait 60 seconds before trying again (like timeout)
    name="PaymentService"    # Give it a name for logging/debugging
)

# Simulate a service function
def call_payment_api(amount):
    """Simulate calling payment API"""
    if amount < 0:
        raise ValueError("Amount cannot be negative")
    return f"Payment of ${amount} processed!"

# Wrap the function with circuit breaker
@cb  # This decorator automatically applies circuit breaker logic
def process_payment_safe(amount):
    """Payment function protected by circuit breaker"""
    return call_payment_api(amount)

# Test it
print("Testing circuit breaker with pybreaker:")
print(f"Initial state: {cb.current_state}")

try:
    result = process_payment_safe(100)
    print(f"✅ {result}")
except Exception as e:
    print(f"❌ Error: {e}")


### Example 2: Using call() Method (Alternative to Decorator)

Instead of decorators, you can use the `call()` method directly:


In [ ]:
# Create circuit breaker
api_cb = CircuitBreaker(fail_max=3, timeout_duration=5, name="ExternalAPI")

# Service function (without decorator)
def fetch_user_data(user_id):
    """Simulate fetching user data from external API"""
    if user_id == 0:
        raise ConnectionError("Database connection failed")
    return f"User {user_id} data retrieved successfully"

# Use call() method to protect the function
print("=== Using call() method ===\n")

for user_id in [1, 2, 3, 0, 0, 0, 4, 5]:
    try:
        # Call the function through circuit breaker
        result = api_cb.call(fetch_user_data, user_id)
        print(f"User {user_id}: ✅ {result}")
        print(f"  Circuit state: {api_cb.current_state}")
    except CircuitBreakerError as e:
        print(f"User {user_id}: 🔴 Circuit OPEN - {e}")
    except Exception as e:
        print(f"User {user_id}: ❌ {e}")
        print(f"  Circuit state: {api_cb.current_state}")
    
    time.sleep(0.3)


### Example 3: Real-World Scenario - Payment Service

Let's recreate the payment service example using pybreaker:


In [ ]:
# Payment service simulation
class PaymentServiceV2:
    def __init__(self):
        self.is_healthy = True
    
    def process_payment(self, amount):
        """Process payment - protected by circuit breaker"""
        if not self.is_healthy:
            raise Exception("Payment service database is down!")
        time.sleep(0.1)  # Simulate processing
        return f"✅ Payment of ${amount} processed successfully"
    
    def set_health(self, healthy):
        self.is_healthy = healthy

# Create circuit breaker for payment service
payment_cb_v2 = CircuitBreaker(
    fail_max=3,
    timeout_duration=3,
    name="PaymentService"
)

payment_service_v2 = PaymentServiceV2()

print("=== Payment Service with pybreaker ===\n")
print("Phase 1: Service is healthy ✅")
print("-" * 50)

for i in range(3):
    try:
        result = payment_cb_v2.call(payment_service_v2.process_payment, 100)
        print(f"Request {i+1}: {result}")
        print(f"  State: {payment_cb_v2.current_state}")
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")
        print(f"  State: {payment_cb_v2.current_state}")

print("\nPhase 2: Service goes down! 😱")
print("-" * 50)
payment_service_v2.set_health(False)

for i in range(5):
    try:
        result = payment_cb_v2.call(payment_service_v2.process_payment, 100)
        print(f"Request {i+1}: {result}")
    except CircuitBreakerError as e:
        print(f"Request {i+1}: 🔴 Circuit OPEN - Service unavailable")
    except Exception as e:
        print(f"Request {i+1}: ❌ {e}")
    print(f"  State: {payment_cb_v2.current_state}")

print("\n⏳ Waiting for timeout...")
time.sleep(4)

print("\nPhase 3: Testing recovery...")
print("-" * 50)
try:
    result = payment_cb_v2.call(payment_service_v2.process_payment, 100)
    print(f"Test: {result}")
except Exception as e:
    print(f"Test: ❌ {e}")
print(f"State: {payment_cb_v2.current_state}")


### Example 4: Monitoring Circuit Breaker States

pybreaker provides easy ways to monitor the circuit breaker:


In [ ]:
# Create a circuit breaker
monitor_cb = CircuitBreaker(fail_max=2, timeout_duration=5, name="MonitorService")

def failing_function():
    raise Exception("Service failed")

# Make some calls and monitor the state
print("=== Monitoring Circuit Breaker ===\n")

for i in range(5):
    print(f"\nCall {i+1}:")
    print(f"  State: {monitor_cb.current_state}")
    print(f"  Fail counter: {monitor_cb.fail_counter}")
    
    try:
        monitor_cb.call(failing_function)
    except CircuitBreakerError:
        print(f"  🔴 Circuit is OPEN")
    except Exception as e:
        print(f"  ❌ {e}")
    
    time.sleep(0.5)

# Check final state
print(f"\n📊 Final Statistics:")
print(f"  Current state: {monitor_cb.current_state}")
print(f"  Fail counter: {monitor_cb.fail_counter}")
print(f"  Success counter: {monitor_cb.success_counter}")

# You can also check if circuit is open
if monitor_cb.current_state == 'open':
    print(f"  ⚠️  Circuit is OPEN - requests will be rejected")
elif monitor_cb.current_state == 'half_open':
    print(f"  🟡 Circuit is HALF_OPEN - testing recovery")
else:
    print(f"  ✅ Circuit is CLOSED - normal operation")


### Example 5: Using with Exception Filtering

You can configure which exceptions should trip the circuit breaker:


In [ ]:
# Create circuit breaker that only fails on specific exceptions
# Only ValueError and ConnectionError will trip the circuit
selective_cb = CircuitBreaker(
    fail_max=3,
    timeout_duration=5,
    name="SelectiveBreaker",
    exclude=[KeyError]  # KeyError won't trip the circuit
)

def selective_function(value):
    """Function that raises different exceptions"""
    if value == "connection_error":
        raise ConnectionError("Connection failed")
    elif value == "key_error":
        raise KeyError("Key not found")
    elif value == "value_error":
        raise ValueError("Invalid value")
    return f"Success: {value}"

print("=== Exception Filtering ===\n")
print("KeyError should NOT trip the circuit:")
print("-" * 40)

# KeyError won't trip the circuit (excluded)
for i in range(3):
    try:
        selective_cb.call(selective_function, "key_error")
    except KeyError:
        print(f"Call {i+1}: KeyError caught (circuit NOT tripped)")
    print(f"  State: {selective_cb.current_state}")

print("\nConnectionError WILL trip the circuit:")
print("-" * 40)
# ConnectionError WILL trip the circuit
for i in range(4):
    try:
        selective_cb.call(selective_function, "connection_error")
    except CircuitBreakerError:
        print(f"Call {i+1}: 🔴 Circuit OPEN (tripped by ConnectionError)")
    except ConnectionError:
        print(f"Call {i+1}: ConnectionError (circuit tripped)")
    print(f"  State: {selective_cb.current_state}")


### Example 6: Decorator vs call() Method Comparison

Both methods work the same way - choose based on your preference:


In [ ]:
# Method 1: Using Decorator (Cleaner for functions you define)
decorator_cb = CircuitBreaker(fail_max=2, timeout_duration=5)

@decorator_cb
def get_user_email(user_id):
    """Function with decorator"""
    if user_id < 0:
        raise ValueError("Invalid user ID")
    return f"user{user_id}@example.com"

# Method 2: Using call() (Better for existing functions/classes)
call_cb = CircuitBreaker(fail_max=2, timeout_duration=5)

def get_user_phone(user_id):
    """Function without decorator"""
    if user_id < 0:
        raise ValueError("Invalid user ID")
    return f"+1-555-{user_id:04d}"

print("=== Decorator Method ===")
try:
    result = get_user_email(1)
    print(f"✅ {result}")
except Exception as e:
    print(f"❌ {e}")

print("\n=== call() Method ===")
try:
    result = call_cb.call(get_user_phone, 2)
    print(f"✅ {result}")
except Exception as e:
    print(f"❌ {e}")

print("\n💡 Tip: Use decorator for your own functions, call() for third-party libraries")


## pybreaker vs Custom Implementation

### Advantages of pybreaker:
- ✅ **Production-ready**: Used in many real-world applications
- ✅ **Thread-safe**: Safe to use in multi-threaded applications
- ✅ **More features**: Callbacks, listeners, state persistence
- ✅ **Well-documented**: Extensive documentation and examples
- ✅ **Maintained**: Regular updates and bug fixes

### When to use pybreaker:
- Production applications
- When you need advanced features (callbacks, listeners)
- Multi-threaded applications
- When you don't want to maintain custom code

### When to use custom implementation:
- Learning purposes (like we did!)
- When you need very specific behavior
- When you want full control over implementation

### Key pybreaker Parameters:
```python
CircuitBreaker(
    fail_max=5,              # Failures before opening (default: 5)
    timeout_duration=60,    # Seconds before retry (default: 60)
    expected_exception=Exception,  # Exceptions that trip circuit
    exclude=[],             # Exceptions that DON'T trip circuit
    name=None               # Name for identification
)
```

### Common Methods:
- `call(func, *args, **kwargs)` - Call function through breaker
- `current_state` - Get current state ('closed', 'open', 'half_open')
- `fail_counter` - Number of failures
- `success_counter` - Number of successes
- `open()` - Manually open circuit
- `close()` - Manually close circuit


## Key Concepts Summary

### 1. **Why Use Circuit Breaker?**
- Prevents cascading failures
- Saves time and resources (no waiting for timeouts)
- Better user experience (fast error responses)
- Automatic recovery when service comes back

### 2. **When to Use?**
- Calling external APIs (payment, email, SMS services)
- Database connections
- Microservices communication
- Any service that can fail and has slow timeouts

### 3. **Configuration Parameters**
- **failure_threshold**: How many failures before opening (e.g., 5)
- **timeout**: How long to wait before testing again (e.g., 60 seconds)
- **half_open_max_calls**: How many test calls in HALF_OPEN (e.g., 3)

### 4. **Important Points**
- Circuit breaker doesn't fix the service - it protects YOUR application
- Different services should have different circuit breakers
- Monitor circuit breaker states to know which services are down
- Can be manually reset if needed

## Real-World Libraries

Instead of building from scratch, you can use:
- **Python**: `circuitbreaker` library, `pybreaker`
- **Java**: `Resilience4j` (mentioned in your notes), `Hystrix`
- **Node.js**: `opossum`, `brakes`
- **Go**: `gobreaker`

## Practice Exercise

Try modifying the code to:
1. Add success threshold (need X successes in HALF_OPEN to go to CLOSED)
2. Add metrics tracking (count how many requests were rejected)
3. Add different timeout strategies (exponential backoff)

---

**Congratulations!** 🎉 You now understand Circuit Breaker pattern!

This is a fundamental pattern used in microservices architecture and distributed systems.
